In [21]:
import pandas as pd

# Load the dataset
df = pd.read_csv("real_time_sensor_data.csv", names=["engine_temp", "battery_voltage", "fuel_pressure", "oil_temp", "engine_load"])

print("Dataset Shape:", df.shape)
print("Columns:", df.columns)
df.head()

Dataset Shape: (1720, 5)
Columns: Index(['engine_temp', 'battery_voltage', 'fuel_pressure', 'oil_temp',
       'engine_load'],
      dtype='object')


,engine_temp,battery_voltage,fuel_pressure,oil_temp,engine_load
0,113.58,12.05,38.38,115.27,67.57
1,101.47,12.26,47.59,104.84,57.23
2,78.58,12.72,33.74,131.49,58.41
3,86.87,14.12,32.86,102.95,22.56
4,86.03,12.44,47.13,118.65,26.16


In [23]:
df['failure_status'] = 0  # Default to no failure

# Define failure conditions based on sensor readings
df.loc[df['engine_temp'] > 110, 'failure_status'] = 1
df.loc[df['battery_voltage'] < 11, 'failure_status'] = 1
df.loc[df['fuel_pressure'] < 30, 'failure_status'] = 1
df.loc[df['oil_temp'] > 130, 'failure_status'] = 1
df.loc[df['engine_load'] > 90, 'failure_status'] = 1

# Check the failure distribution
print(df['failure_status'].value_counts())


failure_status
0    1001
1     719
Name: count, dtype: int64


In [25]:
#from sklearn.ensemble import IsolationForest

#iso_forest = IsolationForest(contamination=0.05, random_state=42)  # Assume 5% failures
#df['failure_status'] = iso_forest.fit_predict(df[['engine_temp', 'battery_voltage', 'fuel_pressure', 'oil_temp', 'engine_load']])

# Convert IsolationForest labels (-1 means anomaly/failure)
#df['failure_status'] = df['failure_status'].apply(lambda x: 1 if x == -1 else 0)

# Check distribution of failures
#print(df['failure_status'].value_counts())


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['failure_status'])  
y = df['failure_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [29]:
# Normalize for models that require it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train multiple models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)  # Train the model
    y_pred = model.predict(X_test)  # Make predictions
    acc = accuracy_score(y_test, y_pred)  # Evaluate accuracy
    
    print(f"{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))


Logistic Regression Accuracy: 0.7994
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       200
           1       0.76      0.76      0.76       144

    accuracy                           0.80       344
   macro avg       0.79      0.79      0.79       344
weighted avg       0.80      0.80      0.80       344

Random Forest Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       144

    accuracy                           1.00       344
   macro avg       1.00      1.00      1.00       344
weighted avg       1.00      1.00      1.00       344

XGBoost Accuracy: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       1.00      1.00      1.00       144

    accuracy                           1.00       344
   macro avg       1.00      1.00  

C:\ProgramData\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:19:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [37]:
!pip install xgboost==3.0.0

  Using cached xgboost-3.0.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.0-py3-none-win_amd64.whl (150.0 MB)


In [41]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Random Forest model with Cross-Validation
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cv_scores = cross_val_score(rf_model, X, y, cv=5)
print("Random Forest Cross-Validation scores:", rf_cv_scores)
print("Average Cross-Validation score:", np.mean(rf_cv_scores))

# XGBoost model with Cross-Validation
xgb_model = XGBClassifier(n_estimators=100, random_state=42)
xgb_cv_scores = cross_val_score(xgb_model, X, y, cv=5)
print("XGBoost Cross-Validation scores:", xgb_cv_scores)
print("Average Cross-Validation score:", np.mean(xgb_cv_scores))


Random Forest Cross-Validation scores: [1.         0.99709302 1.         0.99418605 1.        ]
Average Cross-Validation score: 0.9982558139534884
XGBoost Cross-Validation scores: [0.99418605 1.         0.99709302 1.         1.        ]
Average Cross-Validation score: 0.9982558139534884
